## Protein feature extraction (UniProtKB → cached JSON → descriptors)

This notebook:
- Fetches UniProtKB records (REST, JSON)
- Caches raw per-protein JSON so you don’t re-fetch each run
- Extracts key UniProt annotations into a flat summary
- Computes AAC / PAAC / CTD descriptors for sequences
- Produces a merge-ready `protein_feature_df` indexed by UniProt ID


In [2]:
# Colab/Kaggle environment setup
!pip -q install requests pandas numpy tqdm propy3 biopython

import os
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [3]:
# Config: use a small set of UniProt IDs to verify correctness
# Includes one invalid ID to demonstrate graceful failure handling.
UNIPROT_IDS: List[str] = [
    "P69905",  # Hemoglobin subunit alpha (human)
    "P68871",  # Hemoglobin subunit beta (human)
    "P00734",  # Prothrombin (human)
    "THIS_IS_NOT_A_REAL_ID",
]

CACHE_DIR = Path("cache/uniprot_json")
FEATURES_CSV = Path("cache/protein_features.csv")
METADATA_CSV = Path("cache/protein_metadata_summary.csv")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)


In [4]:
UNIPROT_REST_BASE = "https://rest.uniprot.org/uniprotkb"


def fetch_uniprot_json(uniprot_id: str, *, timeout_s: int = 30, max_retries: int = 3) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """Fetch UniProtKB record JSON via REST.

    Returns (record, error_message). On success: (dict, None). On failure: (None, str).
    """
    url = f"{UNIPROT_REST_BASE}/{uniprot_id}.json"

    last_err: Optional[str] = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=timeout_s, headers={"User-Agent": "ProteinFetch/1.0"})
            if resp.status_code == 200:
                try:
                    return resp.json(), None
                except Exception as e:  # JSON decode
                    return None, f"json_decode_error: {e}"

            if resp.status_code in (404, 400):
                return None, f"http_{resp.status_code}"

            # transient-ish codes
            last_err = f"http_{resp.status_code}"
        except requests.RequestException as e:
            last_err = f"network_error: {e}"

        # exponential backoff
        if attempt < max_retries:
            time.sleep(0.6 * (2 ** (attempt - 1)))

    return None, last_err or "unknown_error"


def _cache_path_for_id(uniprot_id: str) -> Path:
    safe = uniprot_id.replace("/", "_").replace("\\", "_")
    return CACHE_DIR / f"{safe}.json"


def get_uniprot_record_cached(uniprot_id: str) -> Tuple[Optional[Dict[str, Any]], Optional[str], bool]:
    """Load cached UniProt JSON if present; else fetch and cache.

    Returns (record, error_message, from_cache)
    """
    cache_path = _cache_path_for_id(uniprot_id)
    if cache_path.exists():
        try:
            return json.loads(cache_path.read_text(encoding="utf-8")), None, True
        except Exception as e:
            # If cache is corrupted, refetch.
            record, err = fetch_uniprot_json(uniprot_id)
            if record is not None:
                cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
                return record, None, False
            return None, f"cache_read_error_then_{err or e}", False

    record, err = fetch_uniprot_json(uniprot_id)
    if record is not None:
        cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        return record, None, False

    return None, err, False


In [5]:
# Quick smoke-test: fetch a couple records and show cache behavior
failures: List[Dict[str, Any]] = []
records: Dict[str, Dict[str, Any]] = {}

for pid in UNIPROT_IDS:
    rec, err, from_cache = get_uniprot_record_cached(pid)
    if rec is None:
        failures.append({"uniprot_id": pid, "stage": "fetch", "error": err})
        continue
    records[pid] = rec

print(f"Fetched {len(records)}/{len(UNIPROT_IDS)} records")
if failures:
    display(pd.DataFrame(failures))


Fetched 3/4 records


,uniprot_id,stage,error
0,THIS_IS_NOT_A_REAL_ID,fetch,http_400


In [6]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis


def _get(d: Any, path: List[Any], default=None):
    cur = d
    for p in path:
        try:
            if isinstance(p, int):
                cur = cur[p]
            else:
                cur = cur.get(p)
        except Exception:
            return default
        if cur is None:
            return default
    return cur


def _extract_crossref_ids(record: Dict[str, Any], db: str) -> List[str]:
    out: List[str] = []
    for x in record.get("uniProtKBCrossReferences", []) or []:
        if x.get("database") == db:
            pid = x.get("id")
            if pid:
                out.append(pid)
    return out


def _extract_comment_texts(record: Dict[str, Any], comment_type: str) -> List[str]:
    texts: List[str] = []
    for c in record.get("comments", []) or []:
        if c.get("commentType") != comment_type:
            continue
        # UniProt comment schema varies by type; try a few common shapes.
        if "texts" in c:
            for t in c.get("texts", []) or []:
                v = t.get("value") if isinstance(t, dict) else None
                if v:
                    texts.append(v)
        if "text" in c and isinstance(c.get("text"), dict):
            v = c["text"].get("value")
            if v:
                texts.append(v)
        if "subcellularLocations" in c:
            for sl in c.get("subcellularLocations", []) or []:
                loc = _get(sl, ["location", "value"]) or _get(sl, ["location", "id"])
                if loc:
                    texts.append(str(loc))
    return texts


def summarize_uniprot_record(uniprot_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
    seq = _get(record, ["sequence", "value"], "") or ""
    seq_len = _get(record, ["sequence", "length"], None)
    checksum = _get(record, ["sequence", "checksum"], None)

    mw = None
    if seq:
        try:
            mw = float(ProteinAnalysis(seq).molecular_weight())
        except Exception:
            mw = None

    # Names / EC
    rec_name = _get(record, ["proteinDescription", "recommendedName", "fullName", "value"], None)
    ec_numbers: List[str] = []
    for ec in _get(record, ["proteinDescription", "recommendedName", "ecNumbers"], []) or []:
        v = ec.get("value") if isinstance(ec, dict) else None
        if v:
            ec_numbers.append(v)

    # GO terms
    go_terms: List[str] = []
    for go in record.get("uniProtKBCrossReferences", []) or []:
        if go.get("database") == "GO":
            gid = go.get("id")
            if gid:
                go_terms.append(gid)

    # Keywords
    keywords: List[str] = []
    for kw in record.get("keywords", []) or []:
        kid = kw.get("id") or kw.get("value")
        if kid:
            keywords.append(str(kid))

    # Pathways (Reactome)
    reactome_ids = _extract_crossref_ids(record, "Reactome")

    # Feature table counts
    ft = record.get("features", []) or []
    ft_types = [f.get("type") for f in ft if isinstance(f, dict)]
    def _count(t: str) -> int:
        return int(sum(1 for x in ft_types if x == t))

    # Subcellular
    subcell_texts = _extract_comment_texts(record, "SUBCELLULAR LOCATION")

    # Function / Enzyme regulation / Pathway / Tissue specificity
    function_texts = _extract_comment_texts(record, "FUNCTION")
    pathway_texts = _extract_comment_texts(record, "PATHWAY")
    enzyme_reg_texts = _extract_comment_texts(record, "ENZYME REGULATION")
    tissue_texts = _extract_comment_texts(record, "TISSUE SPECIFICITY")
    dev_texts = _extract_comment_texts(record, "DEVELOPMENTAL STAGE")

    pdb_ids = _extract_crossref_ids(record, "PDB")

    # Attempt to count isoforms/variants if present
    # (UniProt encodes isoforms in comments/alternative products; keep best-effort.)
    isoform_count = 0
    alt_products = _get(record, ["comments"], []) or []
    for c in alt_products:
        if isinstance(c, dict) and c.get("commentType") == "ALTERNATIVE PRODUCTS":
            isoform_count += len(c.get("isoforms", []) or [])

    return {
        "uniprot_id": uniprot_id,
        "sequence": seq,
        "sequence_length": seq_len if seq_len is not None else (len(seq) if seq else None),
        "sequence_checksum": checksum,
        "molecular_weight": mw,
        "recommended_name": rec_name,
        "ec_numbers": ";".join(sorted(set(ec_numbers))) if ec_numbers else None,
        "go_terms": ";".join(sorted(set(go_terms))) if go_terms else None,
        "keywords": ";".join(sorted(set(keywords))) if keywords else None,
        "reactome": ";".join(sorted(set(reactome_ids))) if reactome_ids else None,
        "subcellular_location": ";".join(sorted(set(subcell_texts))) if subcell_texts else None,
        "function": " ".join(function_texts) if function_texts else None,
        "pathway": " ".join(pathway_texts) if pathway_texts else None,
        "enzyme_regulation": " ".join(enzyme_reg_texts) if enzyme_reg_texts else None,
        "tissue_specificity": " ".join(tissue_texts) if tissue_texts else None,
        "developmental_stage": " ".join(dev_texts) if dev_texts else None,
        "isoform_count": isoform_count,
        "ft_transmem_count": _count("TRANSMEM"),
        "ft_topo_dom_count": _count("TOPO_DOM"),
        "ft_domain_count": _count("DOMAIN"),
        "ft_region_count": _count("REGION"),
        "ft_binding_count": _count("BINDING"),
        "ft_ptm_count": _count("MOD_RES") + _count("CARBOHYD"),
        "ft_variant_count": _count("VARIANT"),
        "ft_mutagen_count": _count("MUTAGEN"),
        "pdb_count": len(pdb_ids),
        "pdb_ids": ";".join(sorted(set(pdb_ids))) if pdb_ids else None,
    }


In [7]:
# Build a metadata summary DataFrame
metadata_rows: List[Dict[str, Any]] = []

for pid, rec in records.items():
    try:
        metadata_rows.append(summarize_uniprot_record(pid, rec))
    except Exception as e:
        failures.append({"uniprot_id": pid, "stage": "summarize", "error": str(e)})

metadata_df = pd.DataFrame.from_records(metadata_rows).set_index("uniprot_id")
display(metadata_df.head(3))
print("metadata_df shape:", metadata_df.shape)


,sequence,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids
uniprot_id,,,,,,,,,,,,,,,,,,,,,,,,,,
P69905,MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,142,None,15257.3591,Hemoglobin subunit alpha,None,GO:0005344;GO:0005506;GO:0005576;GO:0005615;GO...,KW-0002;KW-0007;KW-0225;KW-0325;KW-0349;KW-036...,R-HSA-1237044;R-HSA-1247673;R-HSA-2168880;R-HS...,None,Involved in oxygen transport from the lung to ...,None,None,Red blood cells,None,0,0,0,0,0,0,0,0,0,341,1A00;1A01;1A0U;1A0Z;1A3N;1A3O;1A9W;1ABW;1ABY;1...
P68871,MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESF...,147,None,15998.2064,Hemoglobin subunit beta,None,GO:0005344;GO:0005576;GO:0005615;GO:0005829;GO...,KW-0002;KW-0007;KW-0225;KW-0325;KW-0349;KW-036...,R-HSA-1237044;R-HSA-1247673;R-HSA-2168880;R-HS...,None,Involved in oxygen transport from the lung to ...,None,None,Red blood cells,None,0,0,0,0,0,0,0,0,0,334,1A00;1A01;1A0U;1A0Z;1A3N;1A3O;1ABW;1ABY;1AJ9;1...
P00734,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,622,None,70036.1126,Prothrombin,3.4.21.5,GO:0001530;GO:0004252;GO:0005102;GO:0005509;GO...,KW-0002;KW-0011;KW-0094;KW-0106;KW-0165;KW-022...,R-HSA-140837;R-HSA-140875;R-HSA-159740;R-HSA-1...,"Secreted, extracellular space","Thrombin, which cleaves bonds after Arg and Ly...",None,None,Expressed by the liver and secreted in plasma,None,0,0,0,0,0,0,0,0,0,400,1A2C;1A3B;1A3E;1A46;1A4W;1A5G;1A61;1ABI;1ABJ;1...


metadata_df shape: (3, 26)


In [8]:
# Descriptor extraction (AAC / PAAC / CTD)
# Use propy3 if available; fall back to simple pure-Python implementations.

AA20 = list("ACDEFGHIKLMNPQRSTVWY")

try:
    from propy import AAComposition
    from propy import CTD as PropyCTD
    from propy import PseudoAAC
    _HAVE_PROPY = True
except Exception:
    _HAVE_PROPY = False


def _aac_fallback(seq: str) -> Dict[str, float]:
    seq = seq.upper()
    n = len(seq)
    if n == 0:
        return {f"aac_{aa}": np.nan for aa in AA20}
    counts = {aa: 0 for aa in AA20}
    for ch in seq:
        if ch in counts:
            counts[ch] += 1
    return {f"aac_{aa}": counts[aa] / n for aa in AA20}


def _paac_fallback(seq: str, lambda_: int = 10, weight: float = 0.05) -> Dict[str, float]:
    """A minimal PAAC-like fallback.

    This is not a full PAAC implementation (which depends on physicochemical property matrices).
    It produces stable numeric features so the pipeline remains runnable if propy3 PAAC is unavailable.
    """
    aac = _aac_fallback(seq)
    # Add a few simple sequence-order proxies (dipeptide frequencies for a subset)
    seq = seq.upper()
    dipep_keys = ["AA", "AC", "CA", "CC", "GG", "PP", "RR", "SS", "TT", "VV"]
    dipep = {f"paac_dipep_{k}": 0.0 for k in dipep_keys}
    if len(seq) >= 2:
        total = len(seq) - 1
        for i in range(total):
            k = seq[i : i + 2]
            if k in dipep_keys:
                dipep[f"paac_dipep_{k}"] += 1.0
        for k in dipep_keys:
            dipep[f"paac_dipep_{k}"] /= total
    # Also carry scaled AAC under PAAC namespace
    paac = {f"paac_{k.replace('aac_', '')}": float(v) for k, v in aac.items()}
    paac.update(dipep)
    paac["paac_lambda"] = float(lambda_)
    paac["paac_weight"] = float(weight)
    return paac


def _ctd_fallback(seq: str) -> Dict[str, float]:
    """CTD fallback using a single common property grouping (hydrophobicity).

    If propy3 CTD is unavailable, this provides a minimal CTD feature vector.
    """
    seq = "".join([c for c in seq.upper() if c.isalpha()])
    if not seq:
        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    # Hydrophobicity (Kyte-Doolittle inspired) 3-group split used in many CTD definitions
    g1 = set("RKEDQN")
    g2 = set("GASTPHY")
    g3 = set("CLVIMFW")

    groups = []
    for aa in seq:
        if aa in g1:
            groups.append(1)
        elif aa in g2:
            groups.append(2)
        elif aa in g3:
            groups.append(3)
        else:
            # Unknown/rare letters -> ignore by skipping
            pass

    if not groups:
        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    n = len(groups)
    c1 = sum(1 for x in groups if x == 1) / n
    c2 = sum(1 for x in groups if x == 2) / n
    c3 = sum(1 for x in groups if x == 3) / n

    return {"ctd_hydro_C1": float(c1), "ctd_hydro_C2": float(c2), "ctd_hydro_C3": float(c3)}


def compute_descriptors(uniprot_id: str, sequence: str) -> Dict[str, float]:
    if not sequence:
        return {"uniprot_id": uniprot_id}

    features: Dict[str, float] = {"uniprot_id": uniprot_id}

    if _HAVE_PROPY:
        try:
            aac = AAComposition.CalculateAAComposition(sequence)
            for k, v in aac.items():
                features[f"aac_{k}"] = float(v)
        except Exception:
            features.update(_aac_fallback(sequence))

        # PAAC (propy)
        try:
            paac = PseudoAAC.GetAPseudoAAC(sequence, lamda=10, weight=0.05)
            for k, v in paac.items():
                features[f"paac_{k}"] = float(v)
        except Exception:
            features.update(_paac_fallback(sequence))

        # CTD (propy)
        try:
            ctd = PropyCTD.CalculateCTD(sequence)
            for k, v in ctd.items():
                features[f"ctd_{k}"] = float(v)
        except Exception:
            features.update(_ctd_fallback(sequence))

        return features

    # Fallbacks only
    features.update(_aac_fallback(sequence))
    features.update(_paac_fallback(sequence))
    features.update(_ctd_fallback(sequence))
    return features


c:\Users\Abdullah\anaconda3\envs\dti_research\lib\site-packages\propy\PseudoAAC.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [9]:
# Compute descriptor features from sequences
feature_rows: List[Dict[str, Any]] = []

for pid in metadata_df.index.tolist():
    seq = metadata_df.loc[pid, "sequence"]
    try:
        feature_rows.append(compute_descriptors(pid, seq))
    except Exception as e:
        failures.append({"uniprot_id": pid, "stage": "descriptors", "error": str(e)})

features_df = pd.DataFrame.from_records(feature_rows).set_index("uniprot_id")
display(features_df.head(3))
print("features_df shape:", features_df.shape)


,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,aac_H,aac_I,aac_L,aac_K,aac_M,aac_F,aac_P,aac_S,aac_T,aac_W,aac_Y,aac_V,paac_APAAC1,paac_APAAC2,paac_APAAC3,paac_APAAC4,paac_APAAC5,paac_APAAC6,paac_APAAC7,paac_APAAC8,paac_APAAC9,paac_APAAC10,paac_APAAC11,paac_APAAC12,paac_APAAC13,paac_APAAC14,paac_APAAC15,paac_APAAC16,paac_APAAC17,paac_APAAC18,paac_APAAC19,paac_APAAC20,paac_APAAC21,paac_APAAC22,paac_APAAC23,paac_APAAC24,paac_APAAC25,paac_APAAC26,paac_APAAC27,paac_APAAC28,paac_APAAC29,paac_APAAC30,paac_APAAC31,paac_APAAC32,paac_APAAC33,paac_APAAC34,paac_APAAC35,paac_APAAC36,paac_APAAC37,paac_APAAC38,paac_APAAC39,paac_APAAC40,ctd__PolarizabilityC1,ctd__PolarizabilityC2,ctd__PolarizabilityC3,ctd__SolventAccessibilityC1,ctd__SolventAccessibilityC2,ctd__SolventAccessibilityC3,ctd__SecondaryStrC1,ctd__SecondaryStrC2,ctd__SecondaryStrC3,ctd__ChargeC1,ctd__ChargeC2,ctd__ChargeC3,ctd__PolarityC1,ctd__PolarityC2,ctd__PolarityC3,ctd__NormalizedVDWVC1,ctd__NormalizedVDWVC2,ctd__NormalizedVDWVC3,ctd__HydrophobicityC1,ctd__HydrophobicityC2,ctd__HydrophobicityC3,ctd__PolarizabilityT12,ctd__PolarizabilityT13,ctd__PolarizabilityT23,ctd__SolventAccessibilityT12,ctd__SolventAccessibilityT13,ctd__SolventAccessibilityT23,ctd__SecondaryStrT12,ctd__SecondaryStrT13,ctd__SecondaryStrT23,ctd__ChargeT12,ctd__ChargeT13,ctd__ChargeT23,ctd__PolarityT12,ctd__PolarityT13,ctd__PolarityT23,ctd__NormalizedVDWVT12,ctd__NormalizedVDWVT13,ctd__NormalizedVDWVT23,ctd__HydrophobicityT12,...,ctd__PolarizabilityD2001,ctd__PolarizabilityD2025,ctd__PolarizabilityD2050,ctd__PolarizabilityD2075,ctd__PolarizabilityD2100,ctd__PolarizabilityD3001,ctd__PolarizabilityD3025,ctd__PolarizabilityD3050,ctd__PolarizabilityD3075,ctd__PolarizabilityD3100,ctd__SolventAccessibilityD1001,ctd__SolventAccessibilityD1025,ctd__SolventAccessibilityD1050,ctd__SolventAccessibilityD1075,ctd__SolventAccessibilityD1100,ctd__SolventAccessibilityD2001,ctd__SolventAccessibilityD2025,ctd__SolventAccessibilityD2050,ctd__SolventAccessibilityD2075,ctd__SolventAccessibilityD2100,ctd__SolventAccessibilityD3001,ctd__SolventAccessibilityD3025,ctd__SolventAccessibilityD3050,ctd__SolventAccessibilityD3075,ctd__SolventAccessibilityD3100,ctd__SecondaryStrD1001,ctd__SecondaryStrD1025,ctd__SecondaryStrD1050,ctd__SecondaryStrD1075,ctd__SecondaryStrD1100,ctd__SecondaryStrD2001,ctd__SecondaryStrD2025,ctd__SecondaryStrD2050,ctd__SecondaryStrD2075,ctd__SecondaryStrD2100,ctd__SecondaryStrD3001,ctd__SecondaryStrD3025,ctd__SecondaryStrD3050,ctd__SecondaryStrD3075,ctd__SecondaryStrD3100,ctd__ChargeD1001,ctd__ChargeD1025,ctd__ChargeD1050,ctd__ChargeD1075,ctd__ChargeD1100,ctd__ChargeD2001,ctd__ChargeD2025,ctd__ChargeD2050,ctd__ChargeD2075,ctd__ChargeD2100,ctd__ChargeD3001,ctd__ChargeD3025,ctd__ChargeD3050,ctd__ChargeD3075,ctd__ChargeD3100,ctd__PolarityD1001,ctd__PolarityD1025,ctd__PolarityD1050,ctd__PolarityD1075,ctd__PolarityD1100,ctd__PolarityD2001,ctd__PolarityD2025,ctd__PolarityD2050,ctd__PolarityD2075,ctd__PolarityD2100,ctd__PolarityD3001,ctd__PolarityD3025,ctd__PolarityD3050,ctd__PolarityD3075,ctd__PolarityD3100,ctd__NormalizedVDWVD1001,ctd__NormalizedVDWVD1025,ctd__NormalizedVDWVD1050,ctd__NormalizedVDWVD1075,ctd__NormalizedVDWVD1100,ctd__NormalizedVDWVD2001,ctd__NormalizedVDWVD2025,ctd__NormalizedVDWVD2050,ctd__NormalizedVDWVD2075,ctd__NormalizedVDWVD2100,ctd__NormalizedVDWVD3001,ctd__NormalizedVDWVD3025,ctd__NormalizedVDWVD3050,ctd__NormalizedVDWVD3075,ctd__NormalizedVDWVD3100,ctd__HydrophobicityD1001,ctd__HydrophobicityD1025,ctd__HydrophobicityD1050,ctd__HydrophobicityD1075,ctd__HydrophobicityD1100,ctd__HydrophobicityD2001,ctd__HydrophobicityD2025,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100
uniprot_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,

features_df shape: (3, 207)


In [10]:
# Combine UniProt summary + descriptors into one ML-ready frame
protein_feature_df = metadata_df.drop(columns=["sequence"]).join(features_df, how="outer")

print("protein_feature_df shape:", protein_feature_df.shape)
missing_by_col = protein_feature_df.isna().sum().sort_values(ascending=False).head(15)
print("Top missing columns:\n", missing_by_col)

display(protein_feature_df.head(3))


protein_feature_df shape: (3, 232)
Top missing columns:
 pathway                           3
sequence_checksum                 3
developmental_stage               3
enzyme_regulation                 3
ec_numbers                        2
subcellular_location              2
ctd__SolventAccessibilityD3050    0
ctd__SecondaryStrD1050            0
ctd__SecondaryStrD1025            0
ctd__SecondaryStrD1001            0
ctd__SolventAccessibilityD3100    0
ctd__SolventAccessibilityD3075    0
sequence_length                   0
ctd__SecondaryStrD1100            0
ctd__SolventAccessibilityD3025    0
dtype: int64


,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,aac_H,aac_I,aac_L,aac_K,aac_M,aac_F,aac_P,aac_S,aac_T,aac_W,aac_Y,aac_V,paac_APAAC1,paac_APAAC2,paac_APAAC3,paac_APAAC4,paac_APAAC5,paac_APAAC6,paac_APAAC7,paac_APAAC8,paac_APAAC9,paac_APAAC10,paac_APAAC11,paac_APAAC12,paac_APAAC13,paac_APAAC14,paac_APAAC15,paac_APAAC16,paac_APAAC17,paac_APAAC18,paac_APAAC19,paac_APAAC20,paac_APAAC21,paac_APAAC22,paac_APAAC23,paac_APAAC24,paac_APAAC25,paac_APAAC26,paac_APAAC27,paac_APAAC28,paac_APAAC29,paac_APAAC30,paac_APAAC31,paac_APAAC32,paac_APAAC33,paac_APAAC34,paac_APAAC35,paac_APAAC36,paac_APAAC37,paac_APAAC38,paac_APAAC39,paac_APAAC40,ctd__PolarizabilityC1,ctd__PolarizabilityC2,ctd__PolarizabilityC3,ctd__SolventAccessibilityC1,ctd__SolventAccessibilityC2,ctd__SolventAccessibilityC3,ctd__SecondaryStrC1,ctd__SecondaryStrC2,ctd__SecondaryStrC3,ctd__ChargeC1,ctd__ChargeC2,ctd__ChargeC3,ctd__PolarityC1,ctd__PolarityC2,ctd__PolarityC3,...,ctd__PolarizabilityD2001,ctd__PolarizabilityD2025,ctd__PolarizabilityD2050,ctd__PolarizabilityD2075,ctd__PolarizabilityD2100,ctd__PolarizabilityD3001,ctd__PolarizabilityD3025,ctd__PolarizabilityD3050,ctd__PolarizabilityD3075,ctd__PolarizabilityD3100,ctd__SolventAccessibilityD1001,ctd__SolventAccessibilityD1025,ctd__SolventAccessibilityD1050,ctd__SolventAccessibilityD1075,ctd__SolventAccessibilityD1100,ctd__SolventAccessibilityD2001,ctd__SolventAccessibilityD2025,ctd__SolventAccessibilityD2050,ctd__SolventAccessibilityD2075,ctd__SolventAccessibilityD2100,ctd__SolventAccessibilityD3001,ctd__SolventAccessibilityD3025,ctd__SolventAccessibilityD3050,ctd__SolventAccessibilityD3075,ctd__SolventAccessibilityD3100,ctd__SecondaryStrD1001,ctd__SecondaryStrD1025,ctd__SecondaryStrD1050,ctd__SecondaryStrD1075,ctd__SecondaryStrD1100,ctd__SecondaryStrD2001,ctd__SecondaryStrD2025,ctd__SecondaryStrD2050,ctd__SecondaryStrD2075,ctd__SecondaryStrD2100,ctd__SecondaryStrD3001,ctd__SecondaryStrD3025,ctd__SecondaryStrD3050,ctd__SecondaryStrD3075,ctd__SecondaryStrD3100,ctd__ChargeD1001,ctd__ChargeD1025,ctd__ChargeD1050,ctd__ChargeD1075,ctd__ChargeD1100,ctd__ChargeD2001,ctd__ChargeD2025,ctd__ChargeD2050,ctd__ChargeD2075,ctd__ChargeD2100,ctd__ChargeD3001,ctd__ChargeD3025,ctd__ChargeD3050,ctd__ChargeD3075,ctd__ChargeD3100,ctd__PolarityD1001,ctd__PolarityD1025,ctd__PolarityD1050,ctd__PolarityD1075,ctd__PolarityD1100,ctd__PolarityD2001,ctd__PolarityD2025,ctd__PolarityD2050,ctd__PolarityD2075,ctd__PolarityD2100,ctd__PolarityD3001,ctd__PolarityD3025,ctd__PolarityD3050,ctd__PolarityD3075,ctd__PolarityD3100,ctd__NormalizedVDWVD1001,ctd__NormalizedVDWVD1025,ctd__NormalizedVDWVD1050,ctd__NormalizedVDWVD1075,ctd__NormalizedVDWVD1100,ctd__NormalizedVDWVD2001,ctd__NormalizedVDWVD2025,ctd__NormalizedVDWVD2050,ctd__NormalizedVDWVD2075,ctd__NormalizedVDWVD2100,ctd__NormalizedVDWVD3001,ctd__NormalizedVDWVD3025,ctd__NormalizedVDWVD3050,ctd__NormalizedVDWVD3075,ctd__NormalizedVDWVD3100,ctd__HydrophobicityD1001,ctd__HydrophobicityD1025,ctd__HydrophobicityD1050,ctd__HydrophobicityD1075,ctd__HydrophobicityD1100,ctd__HydrophobicityD2001,ctd__HydrophobicityD2025,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100
uniprot_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
P69905,142,None,15257.3591,Hemoglobin subunit alpha,None,GO:0005344;GO:0005506;GO:0005576;GO:0005615;GO...,KW-0002;KW-0007;KW-0225;KW-032

## Caching the computed feature table

If `cache/protein_features.csv` exists, you can load it and skip re-fetching UniProt / recomputing descriptors.

- Delete `cache/` if you want a full refresh.
- Raw UniProtKB JSON is stored per ID in `cache/uniprot_json/`.


In [11]:
# Save outputs (and show how to load them)
protein_feature_df.to_csv(FEATURES_CSV)
metadata_df.drop(columns=["sequence"]).to_csv(METADATA_CSV)

print(f"Wrote: {FEATURES_CSV}")
print(f"Wrote: {METADATA_CSV}")

# Example reload
reloaded = pd.read_csv(FEATURES_CSV).set_index("uniprot_id")
print("Reloaded shape:", reloaded.shape)


Wrote: cache\protein_features.csv
Wrote: cache\protein_metadata_summary.csv
Reloaded shape: (3, 232)


In [12]:
# Optional: If you want to skip compute when cached features exist, run this pattern at the top of your notebook.
#
# if FEATURES_CSV.exists():
#     protein_feature_df = pd.read_csv(FEATURES_CSV).set_index("uniprot_id")
# else:
#     # run fetch + summarize + descriptors and then save
#     protein_feature_df.to_csv(FEATURES_CSV)


In [13]:
# Integration snippet: merge protein features into a DTI dataset (KIBA / BindingDB style)
# Expected: your DTI dataframe has a protein identifier column (e.g., "protein_id" containing UniProt IDs)

# Example placeholders (replace with your real dataset path/loader)
# dti_df = pd.read_csv("/path/to/dti.csv")
# dti_df columns might include: ["drug_id", "protein_id", "affinity", ...]

# Merge
# merged_df = dti_df.merge(
#     protein_feature_df.reset_index(),
#     left_on="protein_id",
#     right_on="uniprot_id",
#     how="left",
# )
#
# print("Merged shape:", merged_df.shape)
# print("Unmatched proteins:", merged_df["uniprot_id"].isna().sum())
#
# # Optionally drop rows where protein features are missing:
# # merged_df = merged_df.dropna(subset=["uniprot_id"])


In [14]:
# Error reporting
if failures:
    failures_df = pd.DataFrame(failures).drop_duplicates()
    display(failures_df)
else:
    print("No errors captured.")


,uniprot_id,stage,error
0,THIS_IS_NOT_A_REAL_ID,fetch,http_400


In [15]:
# === Config for structure data (PDB + AlphaFold) ===

PDB_CACHE_DIR = Path("cache/pdb_structures")
ALPHAFOLD_CACHE_DIR = Path("cache/alphafold_structures")
STRUCTURE_FEATURES_CSV = Path("cache/structure_features.csv")
COMPLETE_FEATURES_CSV = Path("cache/complete_protein_features.csv")
STRUCTURE_ERROR_LOG = Path("cache/structure_fetch_errors.json")

PDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ALPHAFOLD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# biopython + requests are already installed above for safety on Colab/Kaggle.

In [16]:
# Extract PDB IDs from UniProt records

def extract_pdb_ids(uniprot_record: Dict[str, Any]) -> List[str]:
    """Return list of PDB IDs referenced in a UniProtKB JSON record."""
    pdbs: List[str] = []
    for x in uniprot_record.get("uniProtKBCrossReferences", []) or []:
        if x.get("database") == "PDB":
            pid = x.get("id")
            if pid:
                pdbs.append(str(pid))
    # De-duplicate while preserving order
    seen = set()
    out: List[str] = []
    for p in pdbs:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out

pdb_map: Dict[str, List[str]] = {}

for pid, rec in records.items():
    pdb_map[pid] = extract_pdb_ids(rec)

pdb_counts_df = pd.DataFrame(
    [
        {"uniprot_id": pid, "n_pdb_structures": len(pdb_ids)}
        for pid, pdb_ids in pdb_map.items()
    ]
).set_index("uniprot_id")

display(pdb_counts_df)


,n_pdb_structures
uniprot_id,
P69905,341
P68871,334
P00734,400


In [17]:
# Download and cache PDB experimental structures

PDB_DOWNLOAD_TEMPLATE = "https://files.rcsb.org/download/{pdb_id}.cif"

structure_failures: List[Dict[str, Any]] = []


def fetch_pdb_structure(pdb_id: str, fmt: str = "cif") -> Optional[str]:
    url = PDB_DOWNLOAD_TEMPLATE.format(pdb_id=pdb_id)
    out_path = PDB_CACHE_DIR / f"{pdb_id}.{fmt}"

    if out_path.exists():
        return str(out_path)

    try:
        resp = requests.get(url, timeout=60, headers={"User-Agent": "ProteinFetch/1.0"})
    except requests.RequestException as e:
        structure_failures.append({"id": pdb_id, "kind": "PDB", "stage": "network", "error": str(e)})
        return None

    if resp.status_code == 404:
        structure_failures.append({"id": pdb_id, "kind": "PDB", "stage": "http", "error": "404"})
        return None
    if resp.status_code != 200:
        structure_failures.append({"id": pdb_id, "kind": "PDB", "stage": "http", "error": str(resp.status_code)})
        return None

    try:
        out_path.write_bytes(resp.content)
        return str(out_path)
    except Exception as e:
        structure_failures.append({"id": pdb_id, "kind": "PDB", "stage": "write", "error": str(e)})
        return None


# Example: download needed PDB files for our small test set
unique_pdb_ids = sorted({p for ids in pdb_map.values() for p in ids})

pdb_file_paths: Dict[str, Optional[str]] = {}

for pid in tqdm(unique_pdb_ids, desc="Downloading PDB CIFs"):
    pdb_file_paths[pid] = fetch_pdb_structure(pid)


In [18]:
# Download and cache AlphaFold predicted structures

ALPHAFOLD_TEMPLATE = "https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.cif"


def fetch_alphafold_structure(uniprot_id: str) -> Optional[str]:
    url = ALPHAFOLD_TEMPLATE.format(uniprot_id=uniprot_id)
    out_path = ALPHAFOLD_CACHE_DIR / f"{uniprot_id}.cif"

    if out_path.exists():
        return str(out_path)

    try:
        resp = requests.get(url, timeout=60, headers={"User-Agent": "ProteinFetch/1.0"})
    except requests.RequestException as e:
        structure_failures.append({"id": uniprot_id, "kind": "AlphaFold", "stage": "network", "error": str(e)})
        return None

    if resp.status_code == 404:
        structure_failures.append({"id": uniprot_id, "kind": "AlphaFold", "stage": "http", "error": "404"})
        return None
    if resp.status_code != 200:
        structure_failures.append({"id": uniprot_id, "kind": "AlphaFold", "stage": "http", "error": str(resp.status_code)})
        return None

    try:
        out_path.write_bytes(resp.content)
        return str(out_path)
    except Exception as e:
        structure_failures.append({"id": uniprot_id, "kind": "AlphaFold", "stage": "write", "error": str(e)})
        return None


alphafold_file_paths: Dict[str, Optional[str]] = {}

for pid in tqdm(metadata_df.index.tolist(), desc="Downloading AlphaFold CIFs"):
    alphafold_file_paths[pid] = fetch_alphafold_structure(pid)

In [19]:
# Compute structure-derived features from CIF files

from Bio.PDB import MMCIFParser
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

try:
    from Bio.PDB.DSSP import DSSP  # Requires external dssp binary; may be unavailable
    _HAVE_DSSP = True
except Exception:
    _HAVE_DSSP = False


def _radius_of_gyration(coords: np.ndarray) -> float:
    if coords.size == 0:
        return float("nan")
    center = coords.mean(axis=0)
    sq_dists = ((coords - center) ** 2).sum(axis=1)
    return float(np.sqrt(sq_dists.mean()))


def _basic_structure_stats(structure) -> Dict[str, Any]:
    chains = list(structure.get_chains())
    n_chains = len(chains)
    residues = [res for res in structure.get_residues() if res.id[0] == " "]
    n_residues = len(residues)

    # Use CA atoms for geometric descriptors when available; fall back to all atoms
    ca_coords = [atom.get_coord() for res in residues for atom in res if atom.get_id() == "CA"]
    if ca_coords:
        coords = np.vstack(ca_coords)
    else:
        all_coords = [atom.get_coord() for atom in structure.get_atoms()]
        coords = np.vstack(all_coords) if all_coords else np.zeros((0, 3))

    rg = _radius_of_gyration(coords)

    # B-factor stats (if present)
    b_factors = [float(atom.get_bfactor()) for atom in structure.get_atoms()]
    if b_factors:
        mean_b = float(np.mean(b_factors))
        std_b = float(np.std(b_factors))
    else:
        mean_b = float("nan")
        std_b = float("nan")

    return {
        "n_residues": n_residues,
        "n_chains": n_chains,
        "radius_gyration": rg,
        "mean_bfactor": mean_b,
        "std_bfactor": std_b,
    }


def _secondary_structure_fractions(structure, cif_path: str) -> Dict[str, Any]:
    """Return helix/sheet/coil percentages.

    Use DSSP if available; otherwise, return NaNs (fallback). DSSP typically not
    available on vanilla Colab/Kaggle, but code is kept for environments where it is.
    """
    if not _HAVE_DSSP:
        return {"helix_pct": float("nan"), "sheet_pct": float("nan"), "coil_pct": float("nan")}

    try:
        model = next(structure.get_models())
        dssp = DSSP(model, cif_path)  # May fail if dssp binary is not installed
    except Exception:
        return {"helix_pct": float("nan"), "sheet_pct": float("nan"), "coil_pct": float("nan")}

    ss_codes = [v[2] for v in dssp]  # DSSP code per residue
    if not ss_codes:
        return {"helix_pct": float("nan"), "sheet_pct": float("nan"), "coil_pct": float("nan")}

    n = len(ss_codes)
    helix = sum(1 for s in ss_codes if s in {"H", "G", "I"}) / n
    sheet = sum(1 for s in ss_codes if s in {"E", "B"}) / n
    coil = 1.0 - helix - sheet
    return {"helix_pct": float(helix), "sheet_pct": float(sheet), "coil_pct": float(coil)}


def _extract_resolution_from_cif(cif_path: str) -> float:
    """Best-effort extraction of resolution (Å) from mmCIF metadata.

    Returns NaN if not available (e.g., AlphaFold models).
    """
    try:
        dct = MMCIF2Dict(cif_path)
    except Exception:
        return float("nan")

    # Common keys: '_refine.ls_d_res_high'
    for key in ["_refine.ls_d_res_high", "_refine_ls_d_res_high"]:
        if key in dct:
            try:
                return float(dct[key][0])
            except Exception:
                continue
    return float("nan")


def compute_structure_features(structure_file: str, source: str) -> Dict[str, Any]:
    """Compute basic structure-derived features from an mmCIF file.

    `source` is one of {"PDB", "AlphaFold"}.
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure_id = Path(structure_file).stem
        structure = parser.get_structure(structure_id, structure_file)
    except Exception as e:
        structure_failures.append({"id": structure_file, "kind": source, "stage": "parse", "error": str(e)})
        return {
            "structure_source": source,
            "n_residues": float("nan"),
            "n_chains": float("nan"),
            "radius_gyration": float("nan"),
            "mean_bfactor": float("nan"),
            "std_bfactor": float("nan"),
            "helix_pct": float("nan"),
            "sheet_pct": float("nan"),
            "coil_pct": float("nan"),
            "pdb_resolution": float("nan"),
        }

    stats = _basic_structure_stats(structure)
    ss = _secondary_structure_fractions(structure, structure_file)

    resolution = float("nan")
    if source == "PDB":
        resolution = _extract_resolution_from_cif(structure_file)

    out = {
        "structure_source": source,
        "pdb_resolution": resolution,
    }
    out.update(stats)
    out.update(ss)
    return out


In [20]:
# Build structure_features_df indexed by uniprot_id

structure_rows: List[Dict[str, Any]] = []

for uniprot_id in metadata_df.index.tolist():
    row: Dict[str, Any] = {
        "uniprot_id": uniprot_id,
        "has_experimental_structure": False,
        "has_alphafold_structure": False,
        "best_pdb_id": None,
        "pdb_file_path": None,
        "alphafold_file_path": alphafold_file_paths.get(uniprot_id),
        "structure_source": "None",
        "pdb_resolution": float("nan"),
    }

    best_pdb_features: Optional[Dict[str, Any]] = None

    # Prefer best-resolution experimental PDB if available
    pdb_ids = pdb_map.get(uniprot_id, [])
    candidates: List[Tuple[float, Dict[str, Any], str]] = []  # (resolution, features, pdb_id)

    for pdb_id in pdb_ids:
        cif_path = pdb_file_paths.get(pdb_id)
        if cif_path is None:
            continue
        feats = compute_structure_features(cif_path, source="PDB")
        res = feats.get("pdb_resolution", float("nan"))
        # Treat NaN as a large value so any real resolution is preferred
        res_sort = res if not np.isnan(res) else 1e6
        candidates.append((res_sort, feats, pdb_id))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        best_res, best_feats, best_pdb_id = candidates[0]
        row["has_experimental_structure"] = True
        row["best_pdb_id"] = best_pdb_id
        row["pdb_file_path"] = pdb_file_paths.get(best_pdb_id)
        row["structure_source"] = "PDB"
        row["pdb_resolution"] = best_feats.get("pdb_resolution", float("nan"))
        row.update(best_feats)
    else:
        # No experimental PDB; fall back to AlphaFold if available
        af_path = alphafold_file_paths.get(uniprot_id)
        if af_path is not None:
            feats = compute_structure_features(af_path, source="AlphaFold")
            row["has_alphafold_structure"] = True
            row["alphafold_file_path"] = af_path
            row["structure_source"] = "AlphaFold"
            row.update(feats)

    structure_rows.append(row)

structure_features_df = pd.DataFrame.from_records(structure_rows).set_index("uniprot_id")

print("structure_features_df shape:", structure_features_df.shape)
display(structure_features_df.head(3))


structure_features_df shape: (3, 15)


,has_experimental_structure,has_alphafold_structure,best_pdb_id,pdb_file_path,alphafold_file_path,structure_source,pdb_resolution,n_residues,n_chains,radius_gyration,mean_bfactor,std_bfactor,helix_pct,sheet_pct,coil_pct
uniprot_id,,,,,,,,,,,,,,,
P69905,True,False,2W72,cache\pdb_structures\2W72.cif,None,PDB,1.07,574,4,23.453003,14.592755,7.625615,NaN,NaN,NaN
P68871,True,False,2W72,cache\pdb_structures\2W72.cif,None,PDB,1.07,574,4,23.453003,14.592755,7.625615,NaN,NaN,NaN
P00734,True,False,5AFY,cache\pdb_structures\5AFY.cif,None,PDB,1.12,291,3,17.385908,19.448777,8.951646,NaN,NaN,NaN


In [21]:
# Persist structure features with cache-aware loading

if STRUCTURE_FEATURES_CSV.exists():
    print(f"Loading cached structure features from {STRUCTURE_FEATURES_CSV}.")
    structure_features_df = pd.read_csv(STRUCTURE_FEATURES_CSV).set_index("uniprot_id")
else:
    print(f"Saving new structure features to {STRUCTURE_FEATURES_CSV}.")
    structure_features_df.to_csv(STRUCTURE_FEATURES_CSV)

# Log structure fetch/parse errors for debugging
if structure_failures:
    try:
        STRUCTURE_ERROR_LOG.write_text(json.dumps(structure_failures, indent=2), encoding="utf-8")
        print(f"Wrote structure error log to {STRUCTURE_ERROR_LOG} (n={len(structure_failures)}).")
    except Exception as e:
        print("Failed to write structure error log:", e)
else:
    print("No structure-related errors captured.")


Saving new structure features to cache\structure_features.csv.
Wrote structure error log to cache\structure_fetch_errors.json (n=3).


In [22]:
# Merge UniProt sequence/features with structure-derived features

complete_features_df = protein_feature_df.join(structure_features_df, how="left")

print("complete_features_df shape:", complete_features_df.shape)

# Basic summary
n_proteins = complete_features_df.shape[0]
exp_struct = complete_features_df["has_experimental_structure"].fillna(False).sum()
af_struct = complete_features_df["has_alphafold_structure"].fillna(False).sum()

print(f"Proteins with experimental PDB structure: {exp_struct}/{n_proteins}")
print(f"Proteins with AlphaFold prediction:      {af_struct}/{n_proteins}")

COMPLETE_FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
complete_features_df.to_csv(COMPLETE_FEATURES_CSV)
print(f"Wrote complete feature matrix to {COMPLETE_FEATURES_CSV}")

display(complete_features_df.head(3))


complete_features_df shape: (3, 247)
Proteins with experimental PDB structure: 3/3
Proteins with AlphaFold prediction:      0/3
Wrote complete feature matrix to cache\complete_protein_features.csv


,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,aac_H,aac_I,aac_L,aac_K,aac_M,aac_F,aac_P,aac_S,aac_T,aac_W,aac_Y,aac_V,paac_APAAC1,paac_APAAC2,paac_APAAC3,paac_APAAC4,paac_APAAC5,paac_APAAC6,paac_APAAC7,paac_APAAC8,paac_APAAC9,paac_APAAC10,paac_APAAC11,paac_APAAC12,paac_APAAC13,paac_APAAC14,paac_APAAC15,paac_APAAC16,paac_APAAC17,paac_APAAC18,paac_APAAC19,paac_APAAC20,paac_APAAC21,paac_APAAC22,paac_APAAC23,paac_APAAC24,paac_APAAC25,paac_APAAC26,paac_APAAC27,paac_APAAC28,paac_APAAC29,paac_APAAC30,paac_APAAC31,paac_APAAC32,paac_APAAC33,paac_APAAC34,paac_APAAC35,paac_APAAC36,paac_APAAC37,paac_APAAC38,paac_APAAC39,paac_APAAC40,ctd__PolarizabilityC1,ctd__PolarizabilityC2,ctd__PolarizabilityC3,ctd__SolventAccessibilityC1,ctd__SolventAccessibilityC2,ctd__SolventAccessibilityC3,ctd__SecondaryStrC1,ctd__SecondaryStrC2,ctd__SecondaryStrC3,ctd__ChargeC1,ctd__ChargeC2,ctd__ChargeC3,ctd__PolarityC1,ctd__PolarityC2,ctd__PolarityC3,...,ctd__SolventAccessibilityD2001,ctd__SolventAccessibilityD2025,ctd__SolventAccessibilityD2050,ctd__SolventAccessibilityD2075,ctd__SolventAccessibilityD2100,ctd__SolventAccessibilityD3001,ctd__SolventAccessibilityD3025,ctd__SolventAccessibilityD3050,ctd__SolventAccessibilityD3075,ctd__SolventAccessibilityD3100,ctd__SecondaryStrD1001,ctd__SecondaryStrD1025,ctd__SecondaryStrD1050,ctd__SecondaryStrD1075,ctd__SecondaryStrD1100,ctd__SecondaryStrD2001,ctd__SecondaryStrD2025,ctd__SecondaryStrD2050,ctd__SecondaryStrD2075,ctd__SecondaryStrD2100,ctd__SecondaryStrD3001,ctd__SecondaryStrD3025,ctd__SecondaryStrD3050,ctd__SecondaryStrD3075,ctd__SecondaryStrD3100,ctd__ChargeD1001,ctd__ChargeD1025,ctd__ChargeD1050,ctd__ChargeD1075,ctd__ChargeD1100,ctd__ChargeD2001,ctd__ChargeD2025,ctd__ChargeD2050,ctd__ChargeD2075,ctd__ChargeD2100,ctd__ChargeD3001,ctd__ChargeD3025,ctd__ChargeD3050,ctd__ChargeD3075,ctd__ChargeD3100,ctd__PolarityD1001,ctd__PolarityD1025,ctd__PolarityD1050,ctd__PolarityD1075,ctd__PolarityD1100,ctd__PolarityD2001,ctd__PolarityD2025,ctd__PolarityD2050,ctd__PolarityD2075,ctd__PolarityD2100,ctd__PolarityD3001,ctd__PolarityD3025,ctd__PolarityD3050,ctd__PolarityD3075,ctd__PolarityD3100,ctd__NormalizedVDWVD1001,ctd__NormalizedVDWVD1025,ctd__NormalizedVDWVD1050,ctd__NormalizedVDWVD1075,ctd__NormalizedVDWVD1100,ctd__NormalizedVDWVD2001,ctd__NormalizedVDWVD2025,ctd__NormalizedVDWVD2050,ctd__NormalizedVDWVD2075,ctd__NormalizedVDWVD2100,ctd__NormalizedVDWVD3001,ctd__NormalizedVDWVD3025,ctd__NormalizedVDWVD3050,ctd__NormalizedVDWVD3075,ctd__NormalizedVDWVD3100,ctd__HydrophobicityD1001,ctd__HydrophobicityD1025,ctd__HydrophobicityD1050,ctd__HydrophobicityD1075,ctd__HydrophobicityD1100,ctd__HydrophobicityD2001,ctd__HydrophobicityD2025,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100,has_experimental_structure,has_alphafold_structure,best_pdb_id,pdb_file_path,alphafold_file_path,structure_source,pdb_resolution,n_residues,n_chains,radius_gyration,mean_bfactor,std_bfactor,helix_pct,sheet_pct,coil_pct
uniprot_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
P69905,142,None,15257.3591,Hemoglobin subunit alpha,None,GO:0005344;GO:0005506;GO:0005576;GO:0005615;GO...,KW-0002;KW-0007;KW-0225;KW-0325;KW-0349;KW-036...,R-HSA-1237044;R-HSA-1247673;R-HSA-2168880;R-HS...,None,Involved in oxygen transport from the lung to ...,None,None,Red blood cells,None,0,0,0,0,0,0,0,0,0,341,1A00;1A0

## Optional: DSSP installation for richer secondary structure features

By default this notebook does **not** require the external `dssp` binary, so it can run on Colab/Kaggle.
If you want more detailed secondary structure annotations and you have `conda` available, you can install DSSP:

```bash
# Optional (requires conda, not available by default on Colab/Kaggle)
!conda install -c salilab dssp
```

After installation, rerun the structure feature cells to populate `helix_pct`, `sheet_pct`, and `coil_pct` using DSSP.

In [23]:
# Updated DTI integration example using complete_features_df

# Example placeholders (replace with your real dataset path/loader)
# dti_df = pd.read_csv("/path/to/dti.csv")
# dti_df columns might include: ["drug_id", "protein_id", "affinity", ...]

# merged_df = dti_df.merge(
#     complete_features_df.reset_index(),
#     left_on="protein_id",
#     right_on="uniprot_id",
#     how="left",
# )
#
# print("Merged shape:", merged_df.shape)
# print("Unmatched proteins (no features):", merged_df["uniprot_id"].isna().sum())
#
# # Show a sample row with all feature groups
# display(merged_df.sample(1).T)
